# Lead Extractor — structured output with confidence & evidence

**Day 2 of the GTM AI sprint.** Turn unstructured text about a person (LinkedIn *About*, an email
signature, a conference bio) into a validated `Lead` — every field carries a **value**, a **confidence**,
and the **evidence** span that supports it. Extraction is a *forced tool call* whose schema is derived
from a Pydantic model; there is no string parsing.

> **Live-run note.** Real extraction needs `ANTHROPIC_API_KEY`. When no key is set this notebook does
> **not** invent model output — it runs the same render/score pipeline against a clearly-labelled
> fixture so the cells still execute, and prints a banner telling you the live path was skipped.
> The deterministic proof that the wiring works lives in `tests/test_lead.py` (`pytest -q`).

In [ ]:
import os, sys
from pathlib import Path

# make the src package and this folder importable when run from the repo
REPO = Path.cwd().parents[1] if Path.cwd().name == 'lead_extractor' else Path.cwd()
sys.path[:0] = [str(REPO / 'src'), str(REPO / 'notebooks' / 'lead_extractor')]

from dotenv import load_dotenv; load_dotenv(REPO / '.env')
from gtm_cli_warmup.cost import cost_tracker
from gtm_cli_warmup.lead import Lead, extract_lead
from synthetic_inputs import SAMPLES

HAS_KEY = bool(os.getenv('ANTHROPIC_API_KEY'))
print(f'{len(SAMPLES)} synthetic inputs loaded | ANTHROPIC_API_KEY present: {HAS_KEY}')

In [ ]:
def show(sample, lead: Lead) -> None:
    """Print a lead as value / confidence / evidence, and score vs gold."""
    print(f"\n=== {sample.id}  ({sample.kind}) ===")
    rows = {
        'full_name': (lead.full_name, None),
        'title': (lead.title, None),
        'seniority': (lead.seniority, sample.gold_seniority),
        'department': (lead.department, sample.gold_department),
        'likely_buying_role': (lead.likely_buying_role, sample.gold_buying_role),
    }
    for name, (fld, gold) in rows.items():
        val = fld.value.value if hasattr(fld.value, 'value') else fld.value
        mark = '' if gold is None else ('  ✓' if str(val) == gold else f'  ✗ (gold={gold})')
        print(f"  {name:<20} {str(val):<26} [{fld.confidence.value:<6}] {mark}")
        print(f"  {'':<20} evidence: {fld.evidence!r}")

def score(pairs) -> None:
    graded = [(s, l) for s, l in pairs]
    hits = sum(
        (l.seniority.value.value == s.gold_seniority)
        + (l.department.value.value == s.gold_department)
        + (l.likely_buying_role.value.value == s.gold_buying_role)
        for s, l in graded
    )
    total = 3 * len(graded)
    print(f"\nEnum accuracy vs gold: {hits}/{total} = {hits/total:.0%}")

In [ ]:
if HAS_KEY:
    pairs = []
    with cost_tracker('extract_lead') as tracker:
        for s in SAMPLES:
            lead, _ = extract_lead(s.text, tracker)
            pairs.append((s, lead)); show(s, lead)
    score(pairs)
    print(f"\nTotal: {tracker.total_tokens} tokens | ${tracker.total_cost_usd:.4f} | {tracker.total_latency_ms} ms")
else:
    print('*** NO API KEY — live extraction skipped. Showing a labelled FIXTURE so cells run. ***')
    print('*** This is NOT a real model call. Set ANTHROPIC_API_KEY and re-run for real output. ***')
    from types import SimpleNamespace as NS
    def f(v, c, e): return NS(value=v, confidence=NS(value=c), evidence=e)
    s = SAMPLES[0]
    fixture = NS(
        full_name=f('(fixture)', 'low', 'render-only stub'),
        title=f('(fixture)', 'low', 'render-only stub'),
        seniority=f('VP', 'low', 'render-only stub'),
        department=f('revenue_operations', 'low', 'render-only stub'),
        likely_buying_role=f('economic_buyer', 'low', 'render-only stub'),
    )
    show(s, fixture)

## What to look for when you run this live

- **Evidence must be verbatim.** Each `evidence` string should be a span you can find in the source
  text. If it's a paraphrase or a fabricated quote, the extractor is drifting — tighten the system prompt.
- **Confidence should track the text.** An explicit title (`sig-01-cro`: *Chief Revenue Officer*) should
  come back `high`; an inferred buying-role on a thin signature should be `low`, not `high`.
- **`unknown` over guessing.** For `likely_buying_role`, the IC analyst (`sig-02-analyst`) is a *user*,
  not a buyer — the model should not promote them to `economic_buyer`.
- **Enum accuracy vs gold** is the headline metric; per-field confidence/evidence is the qualitative check.

The wiring (forced tool use → validated `Lead`, injection-fenced input, cost logged) is proven offline in
`tests/test_lead.py`. This notebook is the human-readable demo on top of it.